In [2]:
import requests
import hmac
from typing import Tuple
from urllib.parse import quote
HOST = "https://bigquant.com"
ACCESS_KEY_HEADER_NAME: str = "X-BigQuant-Access-Key"
ACCESS_SIGNATURE_HEADER_NAME: str = "X-BigQuant-Signature"
ACCESS_TIMESTAMP_HEADER_NAME: str = "X-BigQuant-Timestamp"
STRATEGY_ID = "af08029e-0233-406b-a9eb-030457e420a0"
ACCESS_KEY = "XwX3JordeCqq"
SECRET_KEY = "ksu1NAKTNkBUOD6Y0EQtlfdCeRAtGeeGe4hyqhFmqJnDaeSiPL8Apd3mjPNPMeRR"
# 生成credential
def signature_headers(
    host: str,
    path: str,
    access_key: str,
    secret_key: str,
    body: bytes = b"",
    headers: dict = {},
    params: dict = None,
) -> Tuple[str, dict]:
    """采用aksk给headers添加签名.
    Args:
        host (str): host.
        path (str): 路径.
        access_key (str): ak.
        secret_key (str): sk.
        body (bytes): 请求负载.
        headers (dict, optional): 请求头.
    Returns:
        Tuple[str, dict]: url, 请求头.
    """
    import time
    url = f"{host}{path}"
    if params:
        url = url + "?" + quote("&".join([f"{k}={v}" for k, v in params.items()]))
    path_encode = path.encode()
    timestamp = int(time.time() * 1000)
    timestamp = str(timestamp)
    # 参与签名的消息
    msg = path_encode + body + timestamp.encode()
    signature = hmac.new(
        secret_key.encode(),
        msg=msg,
        digestmod="SHA256",
    ).hexdigest()
    # 给headers这是签名和时间戳
    headers[ACCESS_KEY_HEADER_NAME] = access_key
    headers[ACCESS_SIGNATURE_HEADER_NAME] = signature
    headers[ACCESS_TIMESTAMP_HEADER_NAME] = timestamp
    return url, headers

url = "/bigapis/trading/v1/papertrading/account_performance"
account_performance_params = {
    "strategy_ids": [STRATEGY_ID],
    "fields": ["trading_day", "today_return", "portfolio_value"]
}
account_performance_url, account_performance_headers = signature_headers(HOST, url , ACCESS_KEY, SECRET_KEY)
response = requests.get(url=account_performance_url, headers=account_performance_headers, params=account_performance_params)
response = response.json()["data"]["items"]

In [3]:
import dai
import pandas as pd 
import numpy as np
from datetime import  datetime, timezone
from empyrical import  max_drawdown
strategy = pd.DataFrame(response).rename(columns={0: "trading_day", 1: "today_return", 2: "portfolio_value"}).sort_values("trading_day")
strategy["trading_day"] = pd.to_datetime(strategy["trading_day"])
benchmark = dai.query("select date, close from cn_stock_bar1d where instrument = '000001.SZ' AND date >= '2022-05-26'").df().rename(columns={"date": "trading_day"})
benchmark_first_day = benchmark.iloc[0]["close"]
benchmark["benchmark_today_return"] = (benchmark["close"] - benchmark["close"].shift(1)) / benchmark["close"].shift(1)
strategy = pd.merge(strategy, benchmark, on="trading_day")
strategy.set_index('trading_day', inplace=True)
strategy.dropna(inplace=True)
strategy

,today_return,portfolio_value,close,benchmark_today_return
trading_day,,,,
2022-05-27,0.001886,1001886.13,1587.053486,-0.000705
2022-05-30,-0.007472,994400.14,1575.861289,-0.007052
2022-05-31,0.006521,1000884.57,1584.815046,0.005682
2022-06-01,-0.007509,993369.33,1575.861289,-0.005650
2022-06-02,-0.001267,992110.59,1561.311433,-0.068758
...,...,...,...,...
2024-11-04,0.010602,2045044.14,1464.409797,0.002625
2024-11-05,0.008322,2062063.95,1488.688843,0.016579
2024-11-06,-0.000542,2060947.24,1475.910398,-0.008584


In [4]:
data = {}
# 夏普
def sharpe_ratio(returns, risk_free_rate=0):
    excess_returns = returns.mean() - risk_free_rate
    volatility = returns.std()
    return np.sqrt(252) * excess_returns / volatility if volatility != 0 else float('inf')

# alpha 和 beta
def alpha_beta(returns, benchmark_returns, risk_free_rate=0.0001173):
    # 超额收益
    excess_returns = returns - risk_free_rate
    benchmark_excess_returns = benchmark_returns - risk_free_rate
    
    # 贝塔计算（线性回归）
    ind_residual = benchmark_returns - np.mean(benchmark_returns)
    covariances = np.mean(ind_residual * returns)
    ind_residual = np.square(ind_residual)
    independent_variances = np.mean(ind_residual)
    beta = covariances / independent_variances

    # 阿尔法计算
    alpha = ((excess_returns - benchmark_excess_returns * beta).mean() + 1) ** 252 - 1
    
    return alpha, beta

# 收益波动率
def annual_volatility(returns):
    return returns.std() * np.sqrt(252)  # 假设一年有252个交易日

# 信息比率
def information_ratio(returns, benchmark_returns):
    excess_returns = returns - benchmark_returns
    mean_excess_return = np.mean(excess_returns)
    tracking_error = np.std(excess_returns)
    
    return mean_excess_return / tracking_error if tracking_error != 0 else float('inf')


data["sharp"] = sharpe_ratio(strategy["today_return"])
data["ir"] = information_ratio(strategy["today_return"], strategy["benchmark_today_return"])
data["alpha"], data["beta"] = alpha_beta(strategy["today_return"], strategy["benchmark_today_return"])
data["annual_volatility"] = annual_volatility(strategy["today_return"])
# 信息比率

In [5]:
data

{'sharp': 1.9846456323923314,
 'ir': 0.021193696104421244,
 'alpha': 0.3328850646934136,
 'beta': 0.04989349777073733,
 'annual_volatility': 0.16235891152300433}

In [7]:
import numpy as np

def cal_half_def(returns):
    """
    計算下行風險
    """
    mu = returns.mean()
    temp = returns[returns < mu]
    half_deviation = (sum((temp - mu) ** 2) / len(temp)) ** 0.5
    return half_deviation

def cal_max_continue_loss_days(df):
    """
    計算最大连跌天数
    """
    import numpy as np
    d = np.where(df['today_return']>0, 1, np.where(df['today_return']<0, -1, 0))
    num = 0
    down_days = []
    if d[0] == -1:
        down_days.append(1)
    else:
        down_days.append(0)
    
    for i in range(1, len(d)):
        if d[i] == -1 and d[i-1] == -1:
            num += 1
        elif d[i] == -1 and d[i-1] != -1:
            num = 1
        else:
            num = 0
        down_days.append(num)
    return np.array(down_days).max()

def max_drawdown(ycapital):
    # 計算每日的回撤
    drawdown = []
    tmp_max_capital = ycapital[0]
    for c in ycapital:
        tmp_max_capital = max(c, tmp_max_capital)
        drawdown.append(1 - c / tmp_max_capital)
    
    endidx = np.argmax(drawdown)
    startidx = np.argmax(ycapital[:endidx])
    return round(np.max(drawdown)*100, 2), startidx, endidx

def max_drawdown_restore_time(startidx, endidx, xdate, ycapital):
    """
    计算最大回撤修复期
    """
    maxdd_resore_time = 0    # 要花多少天修复
    restore_endidx = np.inf
    for t in range(endidx, len(xdate)):
        if ycapital[t] >= ycapital[startidx]:
            restore_endidx = t
            break
        else:
            maxdd_resore_time += 1
    restore_endidx = min(restore_endidx, len(xdate) - 1)
    return maxdd_resore_time, restore_endidx

# 年化波动率
annual_volatility = round(strategy['today_return'].std() * np.sqrt(252) * 100, 2)
# 下行风险
downside_risk = round(cal_half_def(strategy["today_return"]) * np.sqrt(252) * 100, 2)
# 最大回撤
max_drawdown_, sd, ed = max_drawdown(strategy['portfolio_value'])
# 最大回撤修复期
maxdd_resore_time, restore_endidx = max_drawdown_restore_time(sd, ed, strategy.index, strategy['portfolio_value'])
# 最大单期跌幅
max_daily_loss = round(strategy['today_return'].min() * 100, 2)
# 最大连跌期数
max_continue_loss_days = cal_max_continue_loss_days(strategy)
# 亏损期数占比
daily_loss_ratio = round(len(strategy[strategy["today_return"] < 0]) / len(strategy), 2)
# 计算回撤开始日期和结束日期
max_drawdown_start = strategy.index[sd]
max_drawdown_end  = strategy.index[ed]

index = ['年化波动率', '下行风险', 
'最大回撤', '最大回撤修复期', 
'最大单期跌幅', '最大连跌期数', 
'亏损期占比', '最大回撤开始日期', 
'最大回撤结束时间']

values = [annual_volatility, downside_risk, max_drawdown_, 
maxdd_resore_time, max_daily_loss, max_continue_loss_days, 
daily_loss_ratio, max_drawdown_start.strftime('%Y-%m-%d'), 
max_drawdown_end.strftime('%Y-%m-%d')]

results = pd.DataFrame(values, index=index, columns=['策略风险指标'])

results

,策略风险指标
年化波动率,16.24
下行风险,15.88
最大回撤,9.34
最大回撤修复期,19
最大单期跌幅,-8.41
最大连跌期数,8
亏损期占比,0.45
最大回撤开始日期,2024-10-08
最大回撤结束时间,2024-10-15


In [ ]:
# 涨跌幅
strategy['net_value'] = (1 + strategy['today_return']).cumprod()
strategy['benchmark_net_value'] = (1 + strategy['benchmark_today_return']).cumprod()

def calculate_price_change(df, key):
    # 从开始到结束的涨跌幅
    total_change = (df[key].iloc[-1] -df[key].iloc[0]) / df[key].iloc[0] * 100

    # 近一月涨跌幅
    last_month = df.last('30D')
    last_month_change = (last_month[key].iloc[-1] - last_month[key].iloc[0]) / last_month[key].iloc[0] * 100

    # 近三月涨跌幅
    last_three_months = df.last('90D')
    last_three_months_change = (last_three_months[key].iloc[-1] - last_three_months[key].iloc[0]) / last_three_months[key].iloc[0] * 100

    # 近半年涨跌幅
    last_six_months = df.last('180D')
    last_six_months_change = (last_six_months[key].iloc[-1] - last_six_months[key].iloc[0]) / last_six_months[key].iloc[0] * 100

    # 近一年涨跌幅
    last_year = df.last('365D')
    last_year_change = (last_year[key].iloc[-1] - last_year[key].iloc[0]) / last_year[key].iloc[0] * 100

    return {
        '总涨跌幅 (%)': total_change,
        '近一月涨跌幅 (%)': last_month_change,
        '近三月涨跌幅 (%)': last_three_months_change,
        '近半年涨跌幅 (%)': last_six_months_change,
        '近一年涨跌幅 (%)': last_year_change,
    }

strategy_raise_and_down = calculate_price_change(strategy, "net_value")
benchmark_raise_and_down = calculate_price_change(strategy, "benchmark_net_value")
strategy_raise_and_down, benchmark_raise_and_down

({'总涨跌幅 (%)': 109.43193599999996,
  '近一月涨跌幅 (%)': 8.924780112115762,
  '近三月涨跌幅 (%)': 12.76582218465781,
  '近半年涨跌幅 (%)': 21.183851185685306,
  '近一年涨跌幅 (%)': 48.72078054772602},
 {'总涨跌幅 (%)': -100.0,
  '近一月涨跌幅 (%)': -70.05356992298556,
  '近三月涨跌幅 (%)': -99.9781559257513,
  '近半年涨跌幅 (%)': -99.99999997830878,
  '近一年涨跌幅 (%)': -100.0})

In [8]:

# 将每日数据转换为每月数据
monthly_df = strategy.resample('M').agg({
    'today_return': lambda x: (1 + x).prod() - 1,
    'benchmark_today_return': lambda x: (1 + x).prod() - 1
})

strategy.reset_index(inplace=True)
monthly_df.reset_index(inplace=True)
months = [strategy.iloc[0]["trading_day"]]

for i in monthly_df["trading_day"].to_list():
    months.append(strategy[strategy["trading_day"] <= i].iloc[-1]["trading_day"])
n = 0
for start, end in zip(months[:-1], months[1:]):
    today_return = strategy[(strategy["trading_day"] > start) & (strategy["trading_day"] <= end)]["today_return"]
    benchmark_today_return = strategy[(strategy["trading_day"] > start) & (strategy["trading_day"] <= end)]['benchmark_today_return']
    excess_returns = today_return - benchmark_today_return
    tracking_error = excess_returns.std()
    mean_excess_returns = excess_returns.mean()
    information_ratio = mean_excess_returns / tracking_error if tracking_error != 0 else 0
    
    volatility = today_return.std()
    
    risk_free_rate = 0.01  # 假设无风险利率为1%
    excess_returns_sharpe = today_return - risk_free_rate / 12
    sharpe_ratio = excess_returns_sharpe.mean() / volatility if volatility != 0 else 0
    monthly_df.at[monthly_df.index[n], 'max_drawdown'] = max_drawdown(today_return)
    monthly_df.at[monthly_df.index[n], 'benchmark_max_drawdown'] = max_drawdown(benchmark_today_return)
    monthly_df.at[monthly_df.index[n], 'tracking_error'] = tracking_error
    monthly_df.at[monthly_df.index[n], 'information_ratio'] = information_ratio
    monthly_df.at[monthly_df.index[n], 'volatility'] = volatility
    monthly_df.at[monthly_df.index[n], 'sharpe_ratio'] = sharpe_ratio
    n += 1
# # 打印结果
monthly_df

KeyError: 0

In [ ]:

# 将每日数据转换为每月数据
yearly_df = strategy.resample('Y').agg({
    'today_return': lambda x: (1 + x).prod() - 1,
    'benchmark_today_return': lambda x: (1 + x).prod() - 1
})

strategy.reset_index(inplace=True)
yearly_df.reset_index(inplace=True)
years = [strategy.iloc[0]["trading_day"]]

for i in yearly_df["trading_day"].to_list():
    years.append(strategy[strategy["trading_day"] <= i].iloc[-1]["trading_day"])
n = 0
for start, end in zip(years[:-1], years[1:]):
    today_return = strategy[(strategy["trading_day"] > start) & (strategy["trading_day"] <= end)]["today_return"]
    benchmark_today_return = strategy[(strategy["trading_day"] > start) & (strategy["trading_day"] <= end)]['benchmark_today_return']
    excess_returns = today_return - benchmark_today_return
    tracking_error = excess_returns.std()
    mean_excess_returns = excess_returns.mean()
    information_ratio = mean_excess_returns / tracking_error if tracking_error != 0 else 0
    
    volatility = today_return.std()
    
    risk_free_rate = 0.01  # 假设无风险利率为1%
    excess_returns_sharpe = today_return - risk_free_rate / 12
    sharpe_ratio = excess_returns_sharpe.mean() / volatility if volatility != 0 else 0
    yearly_df.at[yearly_df.index[n], 'max_drawdown'] = max_drawdown(today_return)
    yearly_df.at[yearly_df.index[n], 'benchmark_max_drawdown'] = max_drawdown(benchmark_today_return)
    yearly_df.at[yearly_df.index[n], 'tracking_error'] = tracking_error
    yearly_df.at[yearly_df.index[n], 'information_ratio'] = information_ratio
    yearly_df.at[yearly_df.index[n], 'volatility'] = volatility
    yearly_df.at[yearly_df.index[n], 'sharpe_ratio'] = sharpe_ratio
    n += 1
# # 打印结果
yearly_df

,trading_day,today_return,benchmark_today_return,max_drawdown,benchmark_max_drawdown,tracking_error,information_ratio,volatility,sharpe_ratio
0,2022-12-31,0.099323,-1.0,-0.072764,-1.0,0.076783,1.207803,0.009001,-0.017493
1,2023-12-31,0.368363,-1.0,-0.055539,-1.0,0.100000,1.380982,0.007973,0.062067
2,2024-12-31,0.392247,-1.0,-0.093426,-1.0,0.070277,2.924418,0.013081,0.066955


In [ ]:
from datetime import datetime
year = strategy.iloc[-1]["trading_day"].year

during_years = [0, 1, 4]

data = {"today_return":[], "benchmark_today_return":[], "max_drawdown":[], "benchmark_max_drawdown":[], "tracking_error":[], "information_ratio":[], "volatility":[], "sharpe_ratio":[]}

for during_year in during_years:
    specific_year = strategy[strategy["trading_day"]> datetime(year-during_year, 1, 1)]
    today_return = (specific_year.iloc[-1]["today_return"] -  specific_year.iloc[0]["today_return"]) / specific_year.iloc[0]["today_return"]
    data["today_return"].append(today_return)
    benchmark_today_return = (specific_year.iloc[-1]["benchmark_today_return"] -  specific_year.iloc[0]["benchmark_today_return"]) / specific_year.iloc[0]["benchmark_today_return"]
    data["benchmark_today_return"].append(benchmark_today_return)
    
    data["max_drawdown"].append(max_drawdown(specific_year["today_return"]))

    data["benchmark_max_drawdown"].append(max_drawdown(specific_year["benchmark_today_return"]))
    
    excess_returns = specific_year["today_return"] - specific_year["benchmark_today_return"]
    data["tracking_error"].append(excess_returns.std())
    mean_excess_returns = excess_returns.mean()
    information_ratio = mean_excess_returns / tracking_error if tracking_error != 0 else 0
    data["information_ratio"].append(information_ratio)
    
    data["volatility"].append(today_return.std())
    
    risk_free_rate = 0.01  # 假设无风险利率为1%
    excess_returns_sharpe = today_return - risk_free_rate / 12
    sharpe_ratio = excess_returns_sharpe.mean() / volatility if volatility != 0 else 0
    data["sharpe_ratio"].append(sharpe_ratio)


data

{'today_return': [1.1642298603811307, 5.662715053558999, inf],
 'benchmark_today_return': [-0.8709009519923058, 2.4383069550143115, -inf],
 'max_drawdown': [-0.09342587579773773,
  -0.0934258757977378,
  -0.0934258757977379],
 'benchmark_max_drawdown': [-1.0, -1.0, -1.0],
 'tracking_error': [0.07027707289090449,
  0.09380521766925623,
  0.09578670371083767],
 'information_ratio': [2.924417879637797,
  2.4038631695777086,
  2.1287678474474783],
 'volatility': [0.0, 0.0, nan],
 'sharpe_ratio': [88.93822503540986, 432.83411876354637, inf]}